## Importar librerías y definir rutas

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import chromadb
from chromadb.config import Settings
import os

# Rutas
METADATA_FOLDER = Path("/home/jupyteruser/work/corpus_upeu/metadatos")
CHUNKS_CSV = METADATA_FOLDER / "chunks.csv"
EMBEDDINGS_NPY = METADATA_FOLDER / "embeddings.npy"
CHUNK_IDS_NPY = METADATA_FOLDER / "chunk_ids.npy"
VECTOR_STORE = Path("/home/jupyteruser/work/vector_store")

# Crear carpeta de la base vectorial
os.makedirs(VECTOR_STORE, exist_ok=True)

## Cargar datos desde archivos

In [2]:
# Cargar metadatos de chunks
df_chunks = pd.read_csv(CHUNKS_CSV)
print(f"Metadatos cargados: {df_chunks.shape[0]} chunks")

# Cargar embeddings (matriz numpy)
embeddings = np.load(EMBEDDINGS_NPY)
print(f"Embeddings cargados: {embeddings.shape}")

# Cargar chunk_ids (permitiendo pickle porque son strings)
chunk_ids = np.load(CHUNK_IDS_NPY, allow_pickle=True)
print(f"Chunk IDs cargados: {len(chunk_ids)}")

# Asegurarse de que el orden coincida
assert len(df_chunks) == embeddings.shape[0] == len(chunk_ids), "Inconsistencia en los datos"

Metadatos cargados: 2488 chunks
Embeddings cargados: (2488, 384)
Chunk IDs cargados: 2488


## Inicializar ChromaDB persistente

In [3]:
# Crear cliente ChromaDB con persistencia en disco
client = chromadb.PersistentClient(path=str(VECTOR_STORE))

# Listar colecciones existentes (por si ya hay una)
colecciones_existentes = client.list_collections()
print(f"Colecciones existentes: {colecciones_existentes}")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


Colecciones existentes: [Collection(name=corpus_upeu)]


## Crear (o resetear) la colección del corpus UPeU

In [4]:
# Nombre de la colección
collection_name = "corpus_upeu"

# Si ya existe, la eliminamos para empezar limpio (en producción, cuidado)
try:
    client.delete_collection(name=collection_name)
    print(f"Colección '{collection_name}' anterior eliminada.")
except:
    pass

# Crear nueva colección
# Especificamos que usaremos embeddings precalculados (no dejamos que Chroma los calcule)
collection = client.create_collection(
    name=collection_name,
    metadata={
        "description": "Corpus de reglamentos universitarios UPeU",
        "hnsw:space": "cosine"
        }
)
print(f"Colección '{collection_name}' creada con espacio 'cosine'.")

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Colección 'corpus_upeu' anterior eliminada.
Colección 'corpus_upeu' creada con espacio 'cosine'.


## Insertar documentos en lotes

In [5]:
# Función auxiliar: definir PRIMERO
def obtener_categoria(doc_name):
    if "estatuto" in doc_name.lower():
        return "A"
    elif "general" in doc_name.lower():
        return "A"
    elif "estudiante" in doc_name.lower():
        return "B"
    elif "investigacion" in doc_name.lower() or "investigación" in doc_name.lower():
        return "C"
    elif "tesis" in doc_name.lower() or "titulo" in doc_name.lower():
        return "C"
    elif "formulario" in doc_name.lower():
        return "C"
    elif "cronograma" in doc_name.lower():
        return "D"
    elif "estudios" in doc_name.lower():
        return "D"
    elif "grado" in doc_name.lower():
        return "D"
    elif "reglamento" in doc_name.lower():
        if "estudiante" in doc_name.lower():
            return "B"
        else:
            return "A"
    else:
        return "E"

# Ahora preparar las listas
ids = df_chunks['chunk_id'].tolist()
documentos = df_chunks['texto'].tolist()
metadatos = df_chunks.apply(
    lambda row: {
        "documento": row['documento'],
        "num_tokens": int(row['num_tokens']),
        "categoria": obtener_categoria(row['documento'])
    },
    axis=1
).tolist()

In [6]:
# Insertar por lotes de 500 chunks
BATCH_SIZE = 500
total = len(ids)
for i in range(0, total, BATCH_SIZE):
    end = min(i+BATCH_SIZE, total)
    collection.add(
        ids=ids[i:end],
        documents=documentos[i:end],
        metadatas=metadatos[i:end],
        embeddings=embeddings[i:end].tolist()  # convertir a lista para ChromaDB
    )
    print(f"Insertados chunks {i} a {end-1} de {total}")

print(f"Inserción completa. Total de documentos en colección: {collection.count()}")

Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


Insertados chunks 0 a 499 de 2488
Insertados chunks 500 a 999 de 2488
Insertados chunks 1000 a 1499 de 2488
Insertados chunks 1500 a 1999 de 2488
Insertados chunks 2000 a 2487 de 2488
Inserción completa. Total de documentos en colección: 2488


## Verificar con una consulta de ejemplo

In [7]:
# Cargar modelo de embeddings (el mismo usado en Notebook 3) para vectorizar la consulta
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# Consulta de prueba
consulta = "¿Cuáles son los requisitos para solicitar titulación?"
print(f"Consulta: {consulta}")

# Generar embedding de la consulta
query_embedding = model.encode([consulta])[0].tolist()

# Buscar en ChromaDB
resultados = collection.query(
    query_embeddings=[query_embedding],
    n_results=3,  # top 3 chunks más relevantes
    include=["documents", "metadatas", "distances"]
)

# Mostrar resultados
print("\nResultados:")
for i, (doc, meta, dist) in enumerate(zip(resultados['documents'][0],
                                          resultados['metadatas'][0],
                                          resultados['distances'][0])):
    print(f"\n--- Resultado {i+1} (distancia: {dist:.4f}) ---")
    print(f"Documento: {meta['documento']}")
    print(f"Categoría: {meta['categoria']}")
    print(f"Fragmento:\n{doc[:500]}...")  # primeros 500 caracteres

/usr/local/lib/python3.10/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
/usr/local/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Consulta: ¿Cuáles son los requisitos para solicitar titulación?


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given



Resultados:

--- Resultado 1 (distancia: 0.3037) ---
Documento: REGLAMENTO DOCENCIA ORDINARIA v3.5
Categoría: A
Fragmento:
. Son requisitos específicos de ingreso en la docencia ordinaria: auxiliar, todas las áreas, factores e ítems obligatorios (O) descritos en la tabla de evaluación N 1 siguiente: ÁREA FACTOR ITEMS I Formación académica profesional 1. Grados 2. Títulos 3. Especialidad/ diplomaturas 4. Estudios 5. Idiomas extranjeros y/o lenguas nativas II Experiencia laboral y profesional 6. Experiencia predocente 7. Experiencia profesional 8. Experiencia en la docencia 9. Experiencia en gestión académica / admini...

--- Resultado 2 (distancia: 0.3092) ---
Documento: REGLAMENTO INTERNO DE TRABAJO Final 2020
Categoría: A
Fragmento:
por el Estatuto y Reglamento de la docencia ordinaria. Artículo 114. Requisitos. El postulante deberá cumplir obligatoriamente, al momento de postular como docente, con los siguientes requisitos: 1. Requerimientos documentarios. Declaración jurada simple 